<span style="font-family: 'Georgia'; font-size: 15px;">
    
## <span style="color:green"> CM2015 Chatbot: University Admissions Guide </span>

**Domain:** Higher Education Admissions Assistance (Germany)


In [1]:
import json
import re
import random
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

# Download required NLTK resources (only needs to be run once)
print("Downloading NLTK data...");
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw-1.4')
print("Downloads complete!")

Downloads complete!


[nltk_data] Downloading package punkt to C:\Users\Grounded
[nltk_data]     Gamer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Grounded
[nltk_data]     Gamer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Grounded
[nltk_data]     Gamer\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Grounded
[nltk_data]     Gamer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Grounded
[nltk_data]     Gamer\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
def load_chatbot_data(intents_path='intents.json', synonyms_path='synonyms.json'):
    """Loads JSON data and constructs regex dictionaries including suggestions."""
    with open(intents_path, 'r') as file:
        data = json.load(file)
        
    rgx2int = {}
    int2res = {}
    int2sug = {} 
    
    for item in data['intents']:
        combined_pattern = r"|".join(item['patterns'])
        compiled_regex = re.compile(combined_pattern, re.IGNORECASE)
        
        rgx2int[compiled_regex] = item['intent']
        int2res[item['intent']] = item['responses']
        int2sug[item['intent']] = item.get('suggestions', [])
        
    with open(synonyms_path, 'r') as file:
        synonyms_dict = json.load(file)
        
    return rgx2int, int2res, int2sug, synonyms_dict

In [3]:
def preprocess_input(user_input, synonyms_dict):
    lemmatizer = WordNetLemmatizer()
    tokens = word_tokenize(user_input)
    
    processed_tokens = []
    for word in tokens:
        word_lower = word.lower()
        
        # 1. Synonym Substitution
        if word_lower in synonyms_dict:
            word_lower = synonyms_dict[word_lower]
            
        # 2. Lemmatization (e.g., "running" -> "run", "better" -> "good")
        # Note: 'v' denotes verb, 'a' denotes adjective
        lemmatized_word = lemmatizer.lemmatize(word_lower, pos='v') 
        lemmatized_word = lemmatizer.lemmatize(lemmatized_word, pos='a')
        
        processed_tokens.append(lemmatized_word)
        
    return " ".join(processed_tokens)

In [4]:
def detect_pattern(processed_input, raw_input, rgx2int, user_state):
    for pattern, intent in rgx2int.items():
        # We search against the RAW input to safely capture proper nouns (names/colors)
        # However, for pure intent matching without variables, you can search against processed_input
        match = pattern.search(raw_input) 
        
        if not match:
            # Fallback: check processed input in case lemmatization is needed to trigger the match
            match = pattern.search(processed_input)
            
        if match:
            # If the regex uses named groups (?P<name>...), extract and store them in memory
            if match.groupdict():
                user_state.update(match.groupdict())
            return intent
            
    return "unknown"

def chatbot_response(intent, int2res, user_state):
    if intent in int2res:
        # Choose a random response from the available list
        selected_response = random.choice(int2res[intent])
        
        # Apply dynamic string substitution (Part 2.6)
        try:
            return selected_response.format(**user_state)
        except KeyError:
            # Failsafe if the template requires a variable not yet in memory
            return selected_response
    else:
        return "I'm not quite sure I understand. Could you rephrase that?"

In [ ]:
def chatbot():
    print("\n=======================================================")
    print("Welcome to the Admissions Guide!")
    print("Type 'quit', 'exit', or 'bye' to exit the chat.")
    print("=======================================================\n")
    
    # Initialize components
    rgx2int, int2res, int2sug, synonyms_dict = load_chatbot_data()
    user_state = {}
    negative_words = ["stressed", "confused", "frustrated", "lost", "hard", "difficult", "overwhelmed", "stuck"]
    
    while True:
        raw_input = input("You: ").strip()
        
        if raw_input.lower() in ["bye", "quit", "exit"]:
            print("Chatbot: Good luck with your applications. Goodbye!")
            break
            
        processed_input = preprocess_input(raw_input, synonyms_dict)
        
        # --- ADVANCED FEATURE: SENTIMENT INTERCEPTOR ---
        is_stressed = any(word in processed_input.split() for word in negative_words)
        
        if is_stressed:
            print("Chatbot: I know navigating university requirements can be overwhelming. Take a deep breath! Which portal or document is causing trouble?")
            print("\n You can ask about: [ uni-assist ] | [ TestAS ] | [ DoSV ]\n")
            continue 
        # -----------------------------------------------
            
        intent = detect_pattern(processed_input, raw_input, rgx2int, user_state)
        response = chatbot_response(intent, int2res, user_state)
        
        print(f"Chatbot: {response}")
        
        # --- ADVANCED FEATURE: CONTEXTUAL SUGGESTIONS ---
        suggestions = int2sug.get(intent, [])
        if suggestions:
            formatted_hints = " | ".join([f"[ {s} ]" for s in suggestions])
            print(f"\n Suggested queries: {formatted_hints}\n")
        # ------------------------------------------------

In [6]:
# Run the chatbot to test
chatbot()


Welcome to the Admissions Guide!
Type 'quit', 'exit', or 'bye' to exit the chat.

Chatbot: Hello! How can I help you with your university application today?

   💡 Suggested queries: [ I want to study in Berlin ] | [ What is DoSV? ] | [ Do I need TestAS? ]

Chatbot: I'm not quite sure I understand. Could you rephrase that?
Chatbot: Programs in czech are highly competitive! Make sure you check if your senior school certificate examination needs to be routed through uni-assist for czech universities.

   💡 Suggested queries: [ How to use uni-assist ] | [ What is the DoSV procedure? ] | [ Exit ]

Chatbot: For uni-assist, you will generally need to upload your senior school certificate examination and language proofs. Make sure to select the specific semester you are targeting.

   💡 Suggested queries: [ Do I need TestAS? ] | [ Exit ]

Chatbot: I'm not quite sure I understand. Could you rephrase that?
Chatbot: TestAS is a central standardized scholastic aptitude test. Many universities hav